# Industrial Defect Detection - Demo Notebook

This notebook demonstrates the usage of the industrial defect detection system.

## Contents
1. Setup and Imports
2. Data Exploration
3. Model Architecture
4. Training Visualization
5. Inference and Results

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

from src.models import UNet, UNetPlusPlus
from src.data import MVTecDataset, get_data_loaders
from src.utils import visualize_predictions, load_checkpoint

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Exploration

In [ ]:
# Load configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Create dataset
dataset = MVTecDataset(
    root_dir=config['data']['root_dir'],
    category=config['data']['category'],
    split='test'
)

print(f'Dataset size: {len(dataset)}')

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    if idx >= len(dataset):
        break
    
    image, mask, label = dataset[idx]
    
    # Convert tensor to numpy
    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.squeeze().numpy()
    
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image_np = std * image_np + mean
    image_np = np.clip(image_np, 0, 1)
    
    # Overlay mask
    if mask_np.sum() > 0:
        overlay = image_np.copy()
        overlay[mask_np > 0.5] = [1, 0, 0]  # Red overlay
        ax.imshow(overlay)
        ax.set_title(f'Sample {idx} - Defect', color='red')
    else:
        ax.imshow(image_np)
        ax.set_title(f'Sample {idx} - Normal', color='green')
    
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
# Create U-Net model
model = UNet(n_channels=3, n_classes=1, base_channels=64).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model size: {total_params * 4 / 1e6:.2f} MB')

In [ ]:
# Test forward pass
dummy_input = torch.randn(1, 3, 256, 256).to(device)

with torch.no_grad():
    output = model(dummy_input)
    
print(f'Input shape: {dummy_input.shape}')
print(f'Output shape: {output.shape}')

## 4. Training Visualization

Load and visualize training metrics from logs.

In [ ]:
import json

# Load metrics
metrics_path = Path('../results/logs/metrics.json')

if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    # Plot metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(metrics['train_loss'], label='Train', linewidth=2)
    axes[0, 0].plot(metrics['val_loss'], label='Val', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice
    axes[0, 1].plot(metrics['train_dice'], label='Train', linewidth=2)
    axes[0, 1].plot(metrics['val_dice'], label='Val', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Coefficient')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # IoU
    axes[1, 0].plot(metrics['train_iou'], label='Train', linewidth=2)
    axes[1, 0].plot(metrics['val_iou'], label='Val', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU Score')
    axes[1, 0].set_title('IoU Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate
    axes[1, 1].plot(metrics['learning_rate'], linewidth=2, color='orange')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print best metrics
    print('\nBest Metrics:')
    print(f'Best Val Loss: {min(metrics["val_loss"]):.4f}')
    print(f'Best Val Dice: {max(metrics["val_dice"]):.4f}')
    print(f'Best Val IoU: {max(metrics["val_iou"]):.4f}')
else:
    print('No training metrics found. Train the model first!')

## 5. Inference and Results

Load trained model and run inference on test images.

In [ ]:
# Load best model
checkpoint_path = Path('../results/checkpoints/best_model.pth')

if checkpoint_path.exists():
    load_checkpoint(model, str(checkpoint_path), device=device)
    print('Model loaded successfully!')
else:
    print('No checkpoint found. Train the model first!')

In [ ]:
# Create data loader
_, val_loader = get_data_loaders(
    root_dir=config['data']['root_dir'],
    category=config['data']['category'],
    batch_size=4,
    num_workers=0
)

# Get a batch
images, masks, labels = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

In [ ]:
# Run inference
model.eval()

with torch.no_grad():
    predictions = model(images)

# Visualize predictions
visualize_predictions(images, masks, predictions, num_samples=4)

In [ ]:
# Calculate metrics for the batch
from src.models.metrics import dice_coefficient, iou_score, precision_recall_f1

dice = dice_coefficient(predictions, masks)
iou = iou_score(predictions, masks)
precision, recall, f1 = precision_recall_f1(predictions, masks)

print(f'\nBatch Metrics:')
print(f'Dice Coefficient: {dice:.4f}')
print(f'IoU Score: {iou:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

## Conclusion

This notebook demonstrated:
- Loading and exploring the MVTec AD dataset
- Creating and inspecting the U-Net model
- Visualizing training progress
- Running inference and evaluating results

For training a new model, use the `train.py` script:
```bash
python train.py --config configs/config.yaml
```

For inference on new images, use the `inference.py` script:
```bash
python inference.py --checkpoint results/checkpoints/best_model.pth --image path/to/image.png
```